# 05 - Clasificación: Riesgo de Abandono

**Pregunta de negocio:** ¿Qué clientes están en riesgo de irse?

## Objetivos
- Predecir qué clientes tienen alto riesgo de abandonar (churn)
- Aprender a manejar **desbalance severo de clases** (pocos casos positivos)
- Comparar técnicas: SMOTE, oversampling manual, scale_pos_weight
- Identificar los factores que más predicen el abandono

## Teoría: Desbalance de Clases

### El problema
Cuando una clase es mucho más rara que la otra (ej. 5% churn vs 95% no churn),
el modelo puede lograr 95% accuracy simplemente prediciendo "no churn" siempre.
Esto es inútil: necesitamos detectar precisamente a ese 5%.

### Soluciones
1. **SMOTE** (Synthetic Minority Over-sampling Technique):
   - Genera muestras *sintéticas* de la clase minoritaria
   - Interpola entre vecinos cercanos en el espacio de features
   - Se aplica SOLO al set de entrenamiento (nunca al test)

2. **Oversampling manual**:
   - Duplicar (con reemplazo) muestras de la clase minoritaria
   - Más simple pero puede causar overfitting

3. **scale_pos_weight** (XGBoost):
   - Penaliza más los errores en la clase minoritaria durante el entrenamiento
   - No modifica los datos, solo la función de pérdida
   - Valor típico: n_negativos / n_positivos

4. **Stratified sampling**:
   - Asegurar que train/test mantengan la proporción original de clases
   - Siempre usar `stratify=y` en `train_test_split`

### Métricas para datos desbalanceados
- **Accuracy** → ENGAÑOSA: no usar como métrica principal
- **Precision-Recall** → MEJOR que ROC para clases desbalanceadas
- **F1** → Resume precision y recall en un solo número
- **AUC-PR** → Área bajo la curva precision-recall

### Costo de negocio del churn
- Adquirir un nuevo cliente cuesta 5-7x más que retener uno existente
- Detectar clientes en riesgo permite intervención proactiva (descuentos, soporte prioritario)
- Un falso negativo (no detectar churn) = pérdida del cliente
- Un falso positivo (predecir churn incorrectamente) = inversión en retención innecesaria (pero el cliente queda más satisfecho)

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, f1_score,
    accuracy_score, precision_score, recall_score, roc_auc_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    HAS_XGBOOST = False
    print("XGBoost no disponible, usando GradientBoostingClassifier como alternativa")

# SMOTE con fallback
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print("imblearn no disponible, se usará oversampling manual como alternativa")

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
data_dir = os.path.join(project_root, "data/raw")
processed_dir = os.path.join(project_root, "data/processed")
models_dir = os.path.join(project_root, "models")
os.makedirs(models_dir, exist_ok=True)
rng = np.random.default_rng(42)

vtype_colors = {'electrico': '#2ecc71', 'gasolina': '#3498db', 'hibrido': '#9b59b6', 'deportivo': '#e74c3c'}

print(f"SMOTE disponible: {HAS_SMOTE}")
print(f"XGBoost nativo: {HAS_XGBOOST}")
print("Librerías cargadas correctamente")

## 1. Carga de datos y creación del target

Definimos **churn_risk** = 1 si:
- `satisfaction_score` <= 2 (insatisfecho o muy insatisfecho)
- AND `would_recommend` == 0 (no recomendaría)

Esta definición captura a los clientes más propensos a irse: insatisfechos que además no recomendarían el servicio.

In [ ]:
# Cargar datos
merged_path = os.path.join(processed_dir, "vehicle_survey_merged.csv")
surveys_path = os.path.join(data_dir, "surveys/buyer_surveys.csv")

if os.path.exists(merged_path):
    df = pd.read_csv(merged_path)
    print(f"Cargado vehicle_survey_merged.csv: {df.shape}")
else:
    # Fallback: cargar encuestas directamente
    df = pd.read_csv(surveys_path)
    print(f"Cargado buyer_surveys.csv: {df.shape}")

print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()

In [ ]:
# Crear variable target: churn_risk
df['churn_risk'] = (
    (df['satisfaction_score'] <= 2) & 
    (df['would_recommend'] == 0)
).astype(int)

# Mostrar desbalance severo
balance = df['churn_risk'].value_counts()
balance_pct = df['churn_risk'].value_counts(normalize=True) * 100

print("Balance de clases (churn_risk):")
print(f"  No churn (0): {balance[0]:,} ({balance_pct[0]:.1f}%)")
print(f"  Churn (1):    {balance.get(1, 0):,} ({balance_pct.get(1, 0):.1f}%)")
print(f"\nRatio de desbalance: {balance[0] / max(balance.get(1, 1), 1):.1f}:1")

# Visualizar desbalance
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

# Barplot del desbalance
colors_churn = ['#2ecc71', '#e74c3c']
axes[0].bar(['No churn (0)', 'Churn (1)'], balance.values, color=colors_churn, edgecolor='white')
for i, v in enumerate(balance.values):
    axes[0].text(i, v + 0.5, f'{v:,}\n({balance_pct.values[i]:.1f}%)', ha='center', fontsize=12)
axes[0].set_ylabel('Número de clientes')
axes[0].set_title('Desbalance de clases: Churn Risk', fontsize=13, fontweight='bold')

# Distribución de satisfaction_score
colors_sat = ['#e74c3c', '#e74c3c', '#f39c12', '#2ecc71', '#2ecc71']
sat_counts = df['satisfaction_score'].value_counts().sort_index()
axes[1].bar(sat_counts.index, sat_counts.values, color=colors_sat, edgecolor='white')
axes[1].set_xlabel('satisfaction_score')
axes[1].set_ylabel('Conteo')
axes[1].set_title('Distribución de Satisfacción', fontsize=13, fontweight='bold')
axes[1].axvline(2.5, color='red', ls='--', alpha=0.7, label='Umbral churn (<=2)')
axes[1].legend()

# Crosstab: satisfaction vs would_recommend
ct = pd.crosstab(df['satisfaction_score'], df['would_recommend'], margins=True)
ct_norm = pd.crosstab(df['satisfaction_score'], df['would_recommend'], normalize='index')
sns.heatmap(ct.iloc[:-1, :-1], annot=True, fmt='d', cmap='YlOrRd', ax=axes[2])
axes[2].set_title('Satisfacción vs Recomendaría', fontsize=13, fontweight='bold')
axes[2].set_xlabel('would_recommend')
axes[2].set_ylabel('satisfaction_score')

plt.tight_layout()
plt.show()

print("\n→ El churn es un evento RARO: pocos clientes tienen satisfacción <=2 Y no recomendarían")
print("→ Esto crea un desbalance severo que engaña a los clasificadores básicos")
print("→ Un modelo 'dummy' que prediga siempre 0 tendría ~{:.0f}% accuracy".format(balance_pct[0]))

In [ ]:
# Preparar features
exclude_cols = ['churn_risk', 'satisfaction_score', 'would_recommend',
                'vehicle_id', 'trip_id', 'customer_id', 'survey_id', 'respondent_id']

# Identificar columnas categóricas y numéricas
cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in exclude_cols]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

print(f"Variables categóricas: {cat_cols}")
print(f"Variables numéricas ({len(num_cols)}): {num_cols}")

# Codificar categóricas
df_model = df.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

feature_cols = [c for c in df_model.columns if c not in exclude_cols]
feature_cols = [c for c in feature_cols if c in num_cols or c in cat_cols]

X = df_model[feature_cols].fillna(0)
y = df_model['churn_risk']

print(f"\nFeatures finales ({len(feature_cols)}): {feature_cols}")
print(f"X shape: {X.shape}")

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} ({y_train.sum()} churn, {y_train.sum()/len(y_train)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} ({y_test.sum()} churn, {y_test.sum()/len(y_test)*100:.1f}%)")

## 2. Baseline SIN balanceo

Primero entrenamos un Random Forest **sin** ningún tratamiento de desbalance para ver el problema.

In [ ]:
# Baseline: Random Forest sin balanceo
rf_baseline = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_pred_base = rf_baseline.predict(X_test)
y_prob_base = rf_baseline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("BASELINE (sin balanceo) - Random Forest")
print("=" * 60)
print(classification_report(y_test, y_pred_base, target_names=['No churn', 'Churn']))

# Confusion matrix
cm_base = confusion_matrix(y_test, y_pred_base)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No churn', 'Churn'], yticklabels=['No churn', 'Churn'])
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Baseline SIN balanceo\n(nótese el bajo recall en Churn)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

recall_base = recall_score(y_test, y_pred_base, zero_division=0)
print(f"\n→ Accuracy alta ({accuracy_score(y_test, y_pred_base):.1%}) pero ENGAÑOSA")
print(f"→ Recall en churn: {recall_base:.1%} — {'¡MUY BAJO! No detecta a los clientes en riesgo' if recall_base < 0.5 else 'Aceptable'}")
print(f"→ El modelo prefiere predecir 'no churn' porque es la clase mayoritaria")
print(f"→ Esto es INÚTIL para el negocio: queremos DETECTAR a los que se van a ir")

## 3. SMOTE: Sobremuestreo Sintético

SMOTE genera muestras *sintéticas* de la clase minoritaria interpolando entre vecinos cercanos.
Se aplica **solo al set de entrenamiento** para no contaminar la evaluación.

In [ ]:
# Aplicar SMOTE (o oversampling manual como fallback)
if HAS_SMOTE:
    print("Usando SMOTE (imblearn)...")
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
else:
    print("Usando oversampling manual (np.random)...")
    # Separar mayoría y minoría
    X_majority = X_train[y_train == 0]
    X_minority = X_train[y_train == 1]
    y_majority = y_train[y_train == 0]
    y_minority = y_train[y_train == 1]
    
    # Sobremuestrear la minoría con reemplazo
    n_to_sample = len(X_majority) - len(X_minority)
    if n_to_sample > 0 and len(X_minority) > 0:
        indices = rng.choice(len(X_minority), size=n_to_sample, replace=True)
        X_minority_up = pd.concat([X_minority, X_minority.iloc[indices]], ignore_index=True)
        y_minority_up = pd.concat([y_minority, y_minority.iloc[indices]], ignore_index=True)
        X_train_bal = pd.concat([X_majority, X_minority_up], ignore_index=True)
        y_train_bal = pd.concat([y_majority, y_minority_up], ignore_index=True)
    else:
        X_train_bal = X_train.copy()
        y_train_bal = y_train.copy()

print(f"\nAntes del balanceo:")
print(f"  Train: {len(X_train)} muestras, {y_train.sum()} churn ({y_train.mean()*100:.1f}%)")
print(f"\nDespués del balanceo:")
print(f"  Train: {len(X_train_bal)} muestras")
if hasattr(y_train_bal, 'value_counts'):
    print(f"  Distribución: {y_train_bal.value_counts().to_dict()}")
else:
    unique, counts = np.unique(y_train_bal, return_counts=True)
    print(f"  Distribución: {dict(zip(unique, counts))}")

# Visualizar antes vs después
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(['No churn', 'Churn'], [sum(y_train == 0), sum(y_train == 1)],
            color=colors_churn, edgecolor='white')
axes[0].set_title('ANTES del balanceo', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Muestras')

if hasattr(y_train_bal, 'value_counts'):
    bal_counts = y_train_bal.value_counts().sort_index()
else:
    unique, counts = np.unique(y_train_bal, return_counts=True)
    bal_counts = pd.Series(counts, index=unique)

axes[1].bar(['No churn', 'Churn'], [bal_counts[0], bal_counts[1]],
            color=colors_churn, edgecolor='white')
axes[1].set_title('DESPUÉS del balanceo (SMOTE/Oversampling)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Muestras')

plt.suptitle('Efecto del balanceo de clases en el set de entrenamiento',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest con datos balanceados
rf_smote = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_bal, y_train_bal)

y_pred_smote = rf_smote.predict(X_test)
y_prob_smote = rf_smote.predict_proba(X_test)[:, 1]

print("=" * 60)
print("RANDOM FOREST + SMOTE/OVERSAMPLING")
print("=" * 60)
print(classification_report(y_test, y_pred_smote, target_names=['No churn', 'Churn']))

In [ ]:
# Comparar matrices de confusión: sin vs con balanceo
cm_smote = confusion_matrix(y_test, y_pred_smote)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['No churn', 'Churn'], yticklabels=['No churn', 'Churn'])
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
recall_b = recall_score(y_test, y_pred_base, zero_division=0)
axes[0].set_title(f'SIN balanceo\nRecall churn: {recall_b:.1%}', fontsize=13, fontweight='bold')

sns.heatmap(cm_smote, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No churn', 'Churn'], yticklabels=['No churn', 'Churn'])
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Real')
recall_s = recall_score(y_test, y_pred_smote, zero_division=0)
axes[1].set_title(f'CON balanceo (SMOTE/Oversampling)\nRecall churn: {recall_s:.1%}', fontsize=13, fontweight='bold')

plt.suptitle('Efecto del balanceo en la detección de churn', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print(f"→ Sin balanceo: recall = {recall_b:.1%} — {'Casi no detecta churn' if recall_b < 0.5 else 'Detección parcial'}")
print(f"→ Con balanceo: recall = {recall_s:.1%} — Mejora significativa en detección")
print(f"→ El trade-off: ganamos recall pero podemos perder algo de precisión (más falsos positivos)")

## 4. XGBoost con scale_pos_weight

Alternativa a SMOTE: en lugar de modificar los datos, le decimos al modelo que los errores
en la clase minoritaria cuestan más. XGBoost tiene el parámetro `scale_pos_weight` para esto.

In [ ]:
# XGBoost con scale_pos_weight
n_neg = sum(y_train == 0)
n_pos = max(sum(y_train == 1), 1)
scale_weight = n_neg / n_pos

print(f"Negativos (train): {n_neg}")
print(f"Positivos (train): {n_pos}")
print(f"scale_pos_weight: {scale_weight:.1f}")

if HAS_XGBOOST:
    xgb_churn = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        scale_pos_weight=scale_weight,
        random_state=42, eval_metric='logloss', verbosity=0
    )
else:
    xgb_churn = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=42
    )

xgb_churn.fit(X_train, y_train)  # Datos originales (sin SMOTE)

y_pred_xgb = xgb_churn.predict(X_test)
y_prob_xgb = xgb_churn.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("XGBOOST (scale_pos_weight)")
print("=" * 60)
print(classification_report(y_test, y_pred_xgb, target_names=['No churn', 'Churn']))

## 5. Comparación de modelos

In [ ]:
# Comparación completa: ROC y Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_eval = [
    ('RF Baseline (sin balanceo)', y_prob_base, '#95a5a6'),
    ('RF + SMOTE/Oversampling', y_prob_smote, '#2ecc71'),
    ('XGBoost (scale_pos_weight)', y_prob_xgb, '#e74c3c'),
]

# Curvas ROC
ax = axes[0]
for name, y_prob, color in models_eval:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name}\n(AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
ax.set_title('Curvas ROC', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')

# Curvas Precision-Recall (MÁS INFORMATIVAS para datos desbalanceados)
ax = axes[1]
for name, y_prob, color in models_eval:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(rec, prec, color=color, lw=2.5, label=f'{name}\n(AP={ap:.3f})')

baseline_rate = y_test.mean()
ax.axhline(baseline_rate, color='gray', ls='--', alpha=0.5, label=f'Baseline ({baseline_rate:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precisión', fontsize=12)
ax.set_title('Curvas Precisión-Recall\n(más informativa para datos desbalanceados)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

print("→ Para datos desbalanceados, la curva Precision-Recall es MÁS informativa que ROC")
print("→ ROC puede ser engañosa porque TN (verdaderos negativos) domina")
print("→ AP (Average Precision) resume la curva PR en un solo número")

In [ ]:
# Tabla resumen
models_summary = []

for name, y_pred, y_prob in [
    ('RF Baseline', y_pred_base, y_prob_base),
    ('RF + SMOTE/Oversampling', y_pred_smote, y_prob_smote),
    ('XGBoost (scale_pos_weight)', y_pred_xgb, y_prob_xgb),
]:
    models_summary.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precisión (churn)': precision_score(y_test, y_pred, zero_division=0),
        'Recall (churn)': recall_score(y_test, y_pred, zero_division=0),
        'F1 (churn)': f1_score(y_test, y_pred, zero_division=0),
        'AUC-ROC': roc_auc_score(y_test, y_prob),
        'AP (PR-AUC)': average_precision_score(y_test, y_prob),
    })

summary_df = pd.DataFrame(models_summary)
print("COMPARACIÓN DE MODELOS - Riesgo de Churn")
print("=" * 90)
print(summary_df.round(4).to_string(index=False))

print("\nNota: Para datos desbalanceados, F1 y AP son las métricas más confiables.")
print("Accuracy es engañosa (un dummy que prediga siempre 0 tendría accuracy alta).")

## 6. Importancia de variables: ¿Qué predice el churn?

In [ ]:
# Feature importance del mejor modelo balanceado
# Usar RF con SMOTE y XGBoost con scale_pos_weight
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(feature_cols) * 0.35)))

# RF + SMOTE
imp_rf = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': rf_smote.feature_importances_
}).sort_values('Importancia', ascending=True)

ax = axes[0]
colors_imp = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(imp_rf)))
ax.barh(imp_rf['Feature'], imp_rf['Importancia'], color=colors_imp)
ax.set_xlabel('Importancia (Gini)')
ax.set_title('RF + SMOTE/Oversampling\nImportancia de Variables', fontsize=13, fontweight='bold')

# XGBoost
imp_xgb = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': xgb_churn.feature_importances_
}).sort_values('Importancia', ascending=True)

ax = axes[1]
colors_imp2 = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(imp_xgb)))
ax.barh(imp_xgb['Feature'], imp_xgb['Importancia'], color=colors_imp2)
ax.set_xlabel('Importancia')
ax.set_title('XGBoost (scale_pos_weight)\nImportancia de Variables', fontsize=13, fontweight='bold')

plt.suptitle('¿Qué variables predicen el abandono de clientes?',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Top features en común
top_rf = set(imp_rf.tail(5)['Feature'].values)
top_xgb = set(imp_xgb.tail(5)['Feature'].values)
common_top = top_rf & top_xgb

print("\nTop 5 features más importantes:")
print(f"  RF + SMOTE:  {list(imp_rf.tail(5)['Feature'].values[::-1])}")
print(f"  XGBoost:     {list(imp_xgb.tail(5)['Feature'].values[::-1])}")
if common_top:
    print(f"  En común:    {list(common_top)}")

print("\n→ Las variables que aparecen en ambos modelos son los predictores más robustos de churn")
print("→ Estas variables son candidatas para monitoreo proactivo de clientes en riesgo")

In [ ]:
# Análisis de las top features: distribución por grupo churn vs no churn
top_features = imp_xgb.tail(4)['Feature'].values[::-1]
n_feats = len(top_features)

if n_feats > 0:
    fig, axes = plt.subplots(1, min(n_feats, 4), figsize=(5 * min(n_feats, 4), 5))
    if n_feats == 1:
        axes = [axes]
    
    for ax, feat in zip(axes, top_features[:4]):
        for label, color, name in [(0, '#2ecc71', 'No churn'), (1, '#e74c3c', 'Churn')]:
            subset = df[df['churn_risk'] == label][feat].dropna()
            if len(subset) > 0:
                ax.hist(subset, bins=30, alpha=0.5, color=color, label=name, density=True)
        ax.set_xlabel(feat)
        ax.set_ylabel('Densidad')
        ax.set_title(f'Distribución: {feat}', fontsize=11, fontweight='bold')
        ax.legend()
    
    plt.suptitle('Top features: Churn vs No Churn', fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()
    
    print("→ Las diferencias visibles entre las distribuciones indican poder predictivo")
    print("→ Variables con distribuciones muy diferentes entre grupos son las más útiles")

## 7. Guardar modelo y resumen

In [ ]:
# Seleccionar mejor modelo por F1 (más apropiado para datos desbalanceados)
best_idx = summary_df['F1 (churn)'].idxmax()
best_model_name = summary_df.loc[best_idx, 'Modelo']
print(f"Mejor modelo por F1: {best_model_name}")

# Seleccionar el objeto del modelo
if 'SMOTE' in best_model_name or 'Oversampling' in best_model_name:
    model_to_save = rf_smote
elif 'XGBoost' in best_model_name:
    model_to_save = xgb_churn
else:
    model_to_save = rf_baseline

# Guardar
model_path = os.path.join(models_dir, "classification_churn_risk.pkl")
model_package = {
    'model': model_to_save,
    'model_name': best_model_name,
    'feature_cols': feature_cols,
    'label_encoders': label_encoders,
    'churn_definition': 'satisfaction_score <= 2 AND would_recommend == 0',
    'balancing_technique': 'SMOTE' if HAS_SMOTE and 'SMOTE' in best_model_name else 'scale_pos_weight' if 'XGBoost' in best_model_name else 'oversampling_manual',
    'metrics': summary_df.loc[best_idx].to_dict(),
}

joblib.dump(model_package, model_path)
print(f"\nModelo guardado en: {model_path}")
print(f"Tamaño: {os.path.getsize(model_path) / 1024:.1f} KB")

# Verificar
loaded = joblib.load(model_path)
print(f"Verificación - Modelo: {loaded['model_name']}")
print(f"Técnica de balanceo: {loaded['balancing_technique']}")

## Resumen

### Hallazgos clave

**1. Desbalance de clases:**
- El churn es un evento raro → desbalance severo en la variable target
- Un modelo sin tratamiento de desbalance tiene accuracy alta pero recall pobre
- Accuracy es una métrica ENGAÑOSA para datos desbalanceados

**2. Técnicas de balanceo:**
- **SMOTE**: genera muestras sintéticas, mejora significativamente el recall
- **scale_pos_weight**: penaliza errores en la minoría, no modifica datos
- Ambas técnicas mejoran la detección de churn a costa de algo de precisión
- Es un trade-off aceptable: mejor investigar una falsa alarma que perder un cliente

**3. Métricas apropiadas:**
- Para datos desbalanceados: F1, AP (Average Precision), curva Precision-Recall
- NO usar accuracy como métrica principal

### Recomendaciones de negocio
- → **Sistema de alertas tempranas**: monitorear las variables predictoras para detectar clientes en riesgo
- → **Intervención proactiva**: ofrecer soporte prioritario, descuentos o mejoras a clientes con alto score de churn
- → **Retención vs adquisición**: invertir en retención es 5-7x más barato que adquirir nuevos clientes
- → **Feedback loop**: registrar qué intervenciones funcionan para mejorar el modelo

### Siguiente notebook
→ `06_model_interpretation.ipynb`: Interpretación de modelos - explicar las predicciones al negocio con permutation importance, PDPs y análisis de predicciones individuales